# PhishingDet – Code & CLI Walkthrough (Updated)

This notebook is a  tutorial of the current repo structure + the CLI commands you can run for **Stage 1 (text-only)**, **Stage 2 (metadata-only)**, and **Stage 3 (hybrid stacking + explanations)**.

> Note: Your current `phishingdet.cli` shows **only** `train` and `predict` in `--help` because that’s what `src/phishingdet/cli.py` currently defines. Stage 2/3 training can still be run via module commands (shown below).


## 1) Repo structure (what goes where)

- `data/raw/`  
  Raw datasets (CSV). Keep large/private datasets here (and usually keep them out of Git).
- `src/phishingdet/`  
  All your **actual project code**.
  - `data/loader.py` → loads dataset CSV + standardises columns
  - `features/` → feature extraction
    - `build_features.py` (Stage 1 TF‑IDF)
    - `build_metadata_features.py` (Stage 2 metadata features)
  - `models/` → training/prediction scripts
    - `train.py`, `predict.py` (Stage 1)
    - `train_metadata.py`, `predict_metadata.py` (Stage 2)
    - `train_hybrid.py`, `predict_hybrid.py` (Stage 3)
- `artifacts/`  
  Saved outputs (models, vectorizers, results JSON, top_features CSV).
  - `artifacts/stage2_metadata/` → Stage 2 outputs
  - `artifacts/stage3_hybrid/` → Stage 3 outputs


## 2) Activate your venv (Windows / macOS)

### Windows (PowerShell)
From the repo root:
```powershell
cd F:\GitHub\fyp-phishing-email-detection
.\.venv\Scripts\Activate.ps1
```
If you see policy issues, set policy once:
```powershell
Set-ExecutionPolicy -Scope CurrentUser RemoteSigned
```

### macOS (zsh)
From the repo root:
```bash
cd ~/GitHub/fyp-phishing-email-detection
source .venv/bin/activate
```

### Quick sanity checks
```bash
python --version
pip --version
python -c "import sklearn, pandas; print('ok')"
```


## 3) Install dependencies + install the package (editable)

From repo root (inside venv):
```bash
pip install -r requirements.txt
pip install -e .
```

Why `pip install -e .`?
- It installs your package **in editable mode**, meaning changes in `src/` are immediately picked up without reinstalling.


## 4) Stage 1 (Text-only) commands

### Train Stage 1
Either run the module:
```bash
python -m phishingdet.models.train
```

Or use the CLI wrapper:
```bash
python -m phishingdet.cli train --test_size 0.2 --random_state 42 --max_features 5000
```

### Predict Stage 1 (module)
```bash
python -m phishingdet.models.predict
```

Artifacts you should see:
- `artifacts/model.joblib`
- `artifacts/vectorizer.joblib`
- `artifacts/results.json` (+ timestamped results)
- `artifacts/top_features.csv`


## 5) Stage 2 (Metadata-only) commands

Stage 2 uses metadata-like features (URL count, IP URL, uppercase ratio, etc.).

### Train Stage 2
```bash
python -m phishingdet.models.train_metadata
```

### Predict Stage 2
```bash
python -m phishingdet.models.predict_metadata
```

Artifacts:
- `artifacts/stage2_metadata/model.joblib`
- `artifacts/stage2_metadata/vectorizer.joblib`
- `artifacts/stage2_metadata/results.json` (+ timestamped)
- `artifacts/stage2_metadata/top_features.csv`
- `artifacts/stage2_metadata/error_analysis.csv`


## 6) Stage 3 (Hybrid stacking) commands

Stage 3 trains a stacking model using **out-of-fold** probabilities from:
- Stage 1 text model
- Stage 2 metadata model

### Train Stage 3
```bash
python -m phishingdet.models.train_hybrid
```

### Predict Stage 3 (module)
```bash
python -m phishingdet.models.predict_hybrid
```

Artifacts:
- `artifacts/stage3_hybrid/` (models + vectorizers + results JSON)


## 7) The main CLI you have now (`phishingdet.cli`)

Run help:
```bash
python -m phishingdet.cli --help
```

You currently have:
- `train` → trains Stage 1 text-only model
- `predict` → runs **hybrid prediction** (Stage 3) and optionally prints an explanation

### Predict with explanation
```bash
python -m phishingdet.cli predict --text "Win a free iPhone now!!! Click here http://1.2.3.4/login" --explain
```

What `--explain` prints (your “reasoning layer”):
- Hybrid probability + decision
- Base model probs (text_prob, meta_prob)
- Triggered metadata features
- Top text cues present (matched against Stage 1 `top_features.csv`)


## 8) Why `--help` only shows `train` and `predict`

Because `cli.py` defines only those subcommands right now.  
Stage 2/Stage 3 training still works through:
- `python -m phishingdet.models.train_metadata`
- `python -m phishingdet.models.train_hybrid`
etc.

If you want later, we can extend `cli.py` to add:
- `train-metadata`, `predict-metadata`
- `train-hybrid`, `predict-hybrid`
so everything is in one CLI.


## 9) Common workflows

### Typical “start work” flow (Windows or Mac)
```bash
# 1) open terminal in repo
# 2) activate venv
# 3) pull latest changes
git pull

# 4) install deps (only if requirements changed)
pip install -r requirements.txt
pip install -e .
```

### After changing code in `src/`
No reinstall needed if you did `pip install -e .`.
Just run the module/CLI again.

### When you add a new library
```bash
pip install <package>
pip freeze > requirements.txt
git add requirements.txt
git commit -m "Add <package>"
git push
```
(Or update requirements manually if you prefer.)
